# Seminário de Ferramentas para Data Science — Selenium em Python

**Disciplina:** Tecnologias para Data Science (DCOMP / UFSJ)  
**Professora:** Prof.ª Carolina Xavier  

## Demonstração

Nesta primeira parte, executamos o navegador em **modo headless** (`--headless=new`) para ilustrar:
1. Configuração de `Options` (modo headless, tamanho de janela e `page_load_strategy`).
2. Sincronização explícita com `WebDriverWait` e `expected_conditions`.
3. Localizadores (`By.CSS_SELECTOR`, `By.XPATH`, `By.ID`, `By.CLASS_NAME`).
4. Interação com elementos (`send_keys`, `clear`, `Keys.ENTER`) pesquisando por **"Ciência da Computação"** e exibindo os resultados.
5. Estruturação dos dados reais coletados do portal da **UFSJ** (`https://www.ufsj.edu.br/mais_noticias.php`) em um `DataFrame` do `pandas`.

In [1]:
import time
from urllib.parse import urljoin
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

### 1.1 Configuração do `WebDriver` e `Options` (Modo Headless)
Define uma função auxiliar para instanciar o `Chrome` com opções recomendadas para coleta estável e reprodutível.

In [2]:
def criar_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1366,768")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.page_load_strategy = "eager"

    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(25)
    return driver

### 1.2 Espera Explícita (`WebDriverWait`) e Coleta das Notícias
Acessa o portal de notícias da UFSJ e extrai **somente** as notícias coletadas (`data`, `titulo` e `url`).

In [3]:
import pandas as pd

URL_UFSJ = "https://www.ufsj.edu.br/mais_noticias.php"

driver = criar_driver(headless=True)
registros_noticias = []

try:
    driver.get(URL_UFSJ)
    wait = WebDriverWait(driver, timeout=15)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.ufsj-corpo-secao ul li a.title")))

    itens_li = driver.find_elements(By.CSS_SELECTOR, "div.ufsj-corpo-secao > ul > li")
    for li in itens_li:
        data_txt = li.find_element(By.CSS_SELECTOR, "span.date").text.strip()
        link_el = li.find_element(By.CSS_SELECTOR, "a.title")
        titulo = " ".join(link_el.text.split())
        href = urljoin(driver.current_url, link_el.get_attribute("href") or "")
        registros_noticias.append({"data": data_txt, "titulo": titulo, "url": href})

finally:
    driver.quit()

df_noticias = pd.DataFrame(registros_noticias)
df_noticias.head(10)

,data,titulo,url
0,20/11/2026,"20 de novembro: vozes que transformam, direito...",https://www.ufsj.edu.br/noticias_ler.php?codig...
1,19/11/2026,UFSJ oferece 16 bolsas para estudantes mães em...,https://www.ufsj.edu.br/noticias_ler.php?codig...
2,17/11/2026,Estudantes de Ciência da Computação conquistam...,https://www.ufsj.edu.br/noticias_ler.php?codig...
3,16/11/2026,"Eventos trazem debates sobre política, formaçã...",https://www.ufsj.edu.br/noticias_ler.php?codig...
4,16/11/2026,UFSJ participa de estudo nacional sobre saúde ...,https://www.ufsj.edu.br/noticias_ler.php?codig...
5,13/11/2026,Evento sobre letramento racial começa na segun...,https://www.ufsj.edu.br/noticias_ler.php?codig...
6,12/11/2026,UFSJ renova parceria com Cátedra UNESCO em Eco...,https://www.ufsj.edu.br/noticias_ler.php?codig...
7,12/11/2026,Residência Médica no HU-UFSJ é aprovado pelo M...,https://www.ufsj.edu.br/noticias_ler.php?codig...
8,11/11/2026,"UFSJ promove seminário sobre território, memór...",https://www.ufsj.edu.br/noticias_ler.php?codig...
9,09/11/2026,Pesquisa da UFSJ é aprovada em programa intern...,https://www.ufsj.edu.br/noticias_ler.php?codig...


### 1.3 Interações com Elementos
Pesquisam por **"Ciência da Computação"** na barra de busca da UFSJ e imprime os 10 primeiros resultados retornados.

In [4]:
driver = criar_driver(headless=True)
registros_busca = []

try:
    driver.get(URL_UFSJ)
    wait = WebDriverWait(driver, timeout=15)

    # Exibe o contêiner da barra de pesquisa (#ufsj-menu-controle) caso esteja oculto
    menu = wait.until(EC.presence_of_element_located((By.ID, "ufsj-menu-controle")))
    driver.execute_script(
        "arguments[0].style.display='block'; arguments[0].style.height='auto'; arguments[0].style.opacity='1';",
        menu,
    )

    campo = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#ufsj-barra-pesquisa input[name='q']")))
    campo.send_keys("termo errado")
    campo.clear()
    campo.send_keys("Ciência da Computação" + Keys.ENTER)

    # Aguarda os resultados do Google Custom Search carregarem na página de busca
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.gsc-webResult.gsc-result a.gs-title")))
    for r in driver.find_elements(By.CSS_SELECTOR, "div.gsc-webResult.gsc-result a.gs-title"):
        titulo = " ".join(r.text.split())
        href = r.get_attribute("href")
        if titulo and href:
            registros_busca.append({"titulo": titulo, "url": href})

finally:
    driver.quit()

df_busca = pd.DataFrame(registros_busca)
df_busca.head(10)

,titulo,url
0,Ciência da Computação - UFSJ,https://ufsj.edu.br/vestibular/ciencia_computa...
1,UFSJ | Universidade Federal de São João del-Rei,https://ufsj.edu.br/ctan/cienciadacomputacao.php
2,Ciência da Computação - UFSJ,https://ufsj.edu.br/portal-repositorio/File/cc...
3,UFSJ | Universidade Federal de São João del-Rei,https://www.ufsj.edu.br/ccomp/documentos.php
4,Índice UFSJ: conheça o curso de Ciência da Com...,https://ufsj.edu.br/noticias_ler.php?codigo_no...
5,dcomp - UFSJ | Universidade Federal de São Joã...,https://ufsj.edu.br/dcomp/
6,Disciplinas Optativas Oferecidas pelo Curso de...,https://ufsj.edu.br/portal-repositorio/File/cc...
7,UFSJ | Universidade Federal de São João del-Rei,https://ufsj.edu.br/ccomp/professores.php
8,CURSO: CIÊNCIA DA COMPUTAÇÃO Turno - UFSJ,https://www.ufsj.edu.br/portal2-repositorio/Fi...
9,Corpo Docente - UFSJ | Universidade Federal de...,https://ufsj.edu.br/cpges/corpo_docente.php


### 1.4 Exploração com Pandas

In [5]:
df_noticias["qtd_caracteres"] = df_noticias["titulo"].str.len()
df_noticias["qtd_palavras"] = df_noticias["titulo"].str.split().str.len()

display(df_noticias.head(10))
display(df_noticias[["qtd_caracteres", "qtd_palavras"]].describe().round(2))

,data,titulo,url,qtd_caracteres,qtd_palavras
0,20/11/2026,"20 de novembro: vozes que transformam, direito...",https://www.ufsj.edu.br/noticias_ler.php?codig...,65,10
1,19/11/2026,UFSJ oferece 16 bolsas para estudantes mães em...,https://www.ufsj.edu.br/noticias_ler.php?codig...,89,12
2,17/11/2026,Estudantes de Ciência da Computação conquistam...,https://www.ufsj.edu.br/noticias_ler.php?codig...,76,10
3,16/11/2026,"Eventos trazem debates sobre política, formaçã...",https://www.ufsj.edu.br/noticias_ler.php?codig...,86,11
4,16/11/2026,UFSJ participa de estudo nacional sobre saúde ...,https://www.ufsj.edu.br/noticias_ler.php?codig...,70,10
5,13/11/2026,Evento sobre letramento racial começa na segun...,https://www.ufsj.edu.br/noticias_ler.php?codig...,52,8
6,12/11/2026,UFSJ renova parceria com Cátedra UNESCO em Eco...,https://www.ufsj.edu.br/noticias_ler.php?codig...,81,12
7,12/11/2026,Residência Médica no HU-UFSJ é aprovado pelo M...,https://www.ufsj.edu.br/noticias_ler.php?codig...,77,12
8,11/11/2026,"UFSJ promove seminário sobre território, memór...",https://www.ufsj.edu.br/noticias_ler.php?codig...,72,9
9,09/11/2026,Pesquisa da UFSJ é aprovada em programa intern...,https://www.ufsj.edu.br/noticias_ler.php?codig...,89,13


,qtd_caracteres,qtd_palavras
count,40.00,40.00
mean,67.82,10.18
std,23.73,3.30
min,19.00,3.00
25%,52.75,8.00
50%,71.00,10.00
75%,83.25,12.25
max,119.00,19.00


---

## Demonstração Visual (Sem `headless`)

Nesta seção **separada**, abre a janela real do Google Chrome (`headless=False`) para navegar por **diferentes páginas** ao vivo durante a apresentação.

In [6]:
def demonstracao_visual_navegador(url: str = "https://www.ufsj.edu.br/mais_noticias.php"):
    options = Options()
    options.add_argument("--start-maximized")
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, timeout=15)

    def destacar_e_clicar(elemento, pausa=1.5):
        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});"
            "arguments[0].style.border = '3px solid red';"
            "arguments[0].style.backgroundColor = '#fff3cd';",
            elemento,
        )
        time.sleep(pausa)
        wait.until(EC.element_to_be_clickable(elemento)).click()

    try:
        # 1. Abre a listagem de notícias (Página 1)
        driver.get(url)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.ufsj-corpo-secao ul li a.title")))
        time.sleep(1.5)

        # 2. Abre a primeira notícia da Página 1
        noticia_1 = driver.find_elements(By.CSS_SELECTOR, "div.ufsj-corpo-secao ul li a.title")[0]
        destacar_e_clicar(noticia_1)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)

        # 3. Volta e navega para a Página 2 da paginação
        driver.back()
        pag_2 = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.ufsj-corpo-secao p a[href*='pagina=2']")))
        destacar_e_clicar(pag_2)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.ufsj-corpo-secao ul li a.title")))
        time.sleep(1.5)

        # 4. Abre uma notícia da Página 2
        noticia_2 = driver.find_elements(By.CSS_SELECTOR, "div.ufsj-corpo-secao ul li a.title")[0]
        destacar_e_clicar(noticia_2)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)

        # 5. Realiza uma busca por 'Ciência da Computação' abrindo a página de resultados
        menu = wait.until(EC.presence_of_element_located((By.ID, "ufsj-menu-controle")))
        driver.execute_script(
            "arguments[0].style.display='block'; arguments[0].style.height='auto'; arguments[0].style.opacity='1';",
            menu,
        )
        campo = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#ufsj-barra-pesquisa input[name='q']")))
        campo.send_keys("Ciência da Computação" + Keys.ENTER)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.gsc-webResult.gsc-result a.gs-title")))
        time.sleep(2.5)

    finally:
        driver.quit()


demonstracao_visual_navegador()